# KuchoLM training data generator

日本語コーパスの原文を MeCab で解析し、`原文 -> NIDA_FICTION` の教師ペアを生成する Colab 用ノートブックです。

現実の韓国語話者の日本語を模倣するものではなく、創作上の語尾スタイルとして扱います。出力は入力文体に関係なく、少し柔らかい口語へ寄せます。


In [ ]:
!pip -q install mecab-python3 unidic-lite datasets

## 1. コーパスを読み込む

既定では Hugging Face `datasets` から日本語テキストを読み込める形にしています。ライセンスを確認した上で `DATASET_NAME` と `TEXT_COLUMN` を変更してください。
ローカルの `.txt` を使う場合は `USE_LOCAL_TEXT = True` にします。

In [ ]:
from pathlib import Path
import json
import re
import MeCab
from datasets import load_dataset

OUTPUT_PATH = Path('/content/kucholm_nida.jsonl')
MAX_ROWS = 100_000

USE_LOCAL_TEXT = False
LOCAL_TEXT_PATH = Path('/content/corpus.txt')

DATASET_NAME = 'range3/cc100-ja'
DATASET_SPLIT = 'train'
TEXT_COLUMN = 'text'

tagger = MeCab.Tagger()


In [ ]:
if USE_LOCAL_TEXT:
    corpus = (line.strip() for line in LOCAL_TEXT_PATH.open(encoding='utf-8'))
else:
    dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT, streaming=True)
    corpus = (str(row[TEXT_COLUMN]).strip() for row in dataset)


## 2. MeCab で文末を解析して変換

丁寧語をそのまま硬い常体へ落とすのではなく、軽く口語化してから創作語尾を付けます。

例:
- `行きます` -> `行くニダよ`
- `行きました` -> `行ったニダよ`
- `ありません` -> `ないニダね`
- `でした` -> `だったニダよ`
- `〜んです` -> `〜んニダよ`
- 疑問文 -> `ニカ？`

文末の `よ` / `ね` は決め打ちしすぎないよう、元文の雰囲気に応じて使い分けます。

In [ ]:
URL_RE = re.compile(r'https?://|www\.|```|`[^`]+`')
END_PUNCT = {'。', '！', '!', '？', '?'}

def parse_tokens(text):
    node = tagger.parseToNode(text)
    tokens = []
    while node:
        if node.surface:
            features = node.feature.split(',')
            tokens.append({
                'surface': node.surface,
                'pos': features[0] if len(features) > 0 else '',
                'ctype': features[4] if len(features) > 4 else '*',
                'lemma': features[7] if len(features) > 7 else '*',
                'orth_base': features[10] if len(features) > 10 else '*',
            })
        node = node.next
    return tokens

def dictionary_form(token):
    for key in ('orth_base', 'lemma'):
        value = token.get(key, '*')
        if value not in {'', '*'} and re.search(r'[ぁ-ん一-龯]', value):
            return value
    return token['surface']

def is_ichidan(token, base):
    ctype = token.get('ctype', '')
    if '下一段' in ctype or '上一段' in ctype or '一段' in ctype:
        return True
    return base.endswith('る') and token.get('surface', '') == base[:-1]

def ta_form(base, token):
    if base == '行く': return '行った'
    if base == '来る': return '来た'
    if base == 'する': return 'した'
    if is_ichidan(token, base): return base[:-1] + 'た' if base.endswith('る') else base + 'た'
    if base.endswith(('う', 'つ', 'る')): return base[:-1] + 'った'
    if base.endswith(('む', 'ぶ', 'ぬ')): return base[:-1] + 'んだ'
    if base.endswith('く'): return base[:-1] + 'いた'
    if base.endswith('ぐ'): return base[:-1] + 'いだ'
    if base.endswith('す'): return base[:-1] + 'した'
    return base + 'た'

def nai_form(base, token):
    if base == 'する': return 'しない'
    if base == '来る': return '来ない'
    if is_ichidan(token, base): return base[:-1] + 'ない' if base.endswith('る') else base + 'ない'
    if base.endswith('う'): return base[:-1] + 'わない'
    mapping = {'く':'か','ぐ':'が','す':'さ','つ':'た','ぬ':'な','ぶ':'ば','む':'ま','る':'ら'}
    last = base[-1:]
    return base[:-1] + mapping[last] + 'ない' if last in mapping else base + 'ない'

def find_last_verb(tokens, before_index):
    for index in range(before_index - 1, -1, -1):
        if tokens[index]['pos'] == '動詞':
            return index, tokens[index]
    return None, None

def soft_ending(original, is_question=False):
    if is_question:
        return 'ニカ'
    if re.search(r'(ね|よ|な|かな|かも|けど|けどね)$', original):
        return 'ニダ'
    if re.search(r'(ない|ません|難しい|心配|残念|大丈夫)$', original):
        return 'ニダね'
    return 'ニダよ'

def soften_surface(text):
    rules = [
        (r'ということです$', 'ってこと'),
        (r'ということでした$', 'ってことだった'),
        (r'のであります$', 'んだ'),
        (r'であります$', 'なんだ'),
        (r'なのです$', 'なんだ'),
        (r'のです$', 'んだ'),
        (r'でしょう$', 'だろう'),
        (r'ではありません$', 'じゃない'),
        (r'ではないです$', 'じゃない'),
        (r'ではない$', 'じゃない'),
    ]
    for pattern, replacement in rules:
        text = re.sub(pattern, replacement, text)
    return text

def to_nida(text):
    text = text.strip()
    if not text or URL_RE.search(text):
        return None

    punctuation = text[-1] if text[-1:] in END_PUNCT else ''
    body = text[:-1] if punctuation else text
    body = soften_surface(body)
    tokens = parse_tokens(body)
    if not tokens:
        return None

    is_question = punctuation in {'？', '?'}
    if tokens and tokens[-1]['surface'] == 'か':
        tokens.pop()
        is_question = True

    surfaces = [token['surface'] for token in tokens]
    ending = soft_ending(body, is_question)
    tail = punctuation or ('？' if is_question else '')

    patterns = [
        (['ませ', 'ん', 'でし', 'た'], 'negative_past'),
        (['ませ', 'ん'], 'negative'),
        (['まし', 'た'], 'past'),
        (['ます'], 'present'),
    ]

    for suffix, mode in patterns:
        if len(surfaces) < len(suffix) or surfaces[-len(suffix):] != suffix:
            continue
        verb_index, verb = find_last_verb(tokens, len(tokens) - len(suffix))
        if verb is None:
            continue
        base = dictionary_form(verb)
        prefix = ''.join(token['surface'] for token in tokens[:verb_index])
        if mode == 'present': replacement = base
        elif mode == 'past': replacement = ta_form(base, verb)
        else:
            negative = nai_form(base, verb)
            replacement = negative if mode == 'negative' else negative[:-2] + 'なかった'
        return prefix + replacement + ending + tail

    if surfaces[-2:] == ['でし', 'た']:
        return ''.join(surfaces[:-2]) + 'だった' + ending + tail
    if surfaces[-1:] == ['です']:
        return ''.join(surfaces[:-1]) + ending + tail

    joined = ''.join(surfaces)
    if joined.endswith(('ね', 'よ', 'な')):
        particle = joined[-1]
        joined = joined[:-1]
        return joined + 'ニダ' + particle + tail
    return joined + ending + tail


## 3. 変換テスト

In [ ]:
tests = [
    '今日は学校です。',
    '明日は学校に行きます。',
    '昨日は学校に行きました。',
    '魚を食べました。',
    '本を読みません。',
    'これは本当ですか？',
    'この方法は適切ではありません。',
    'これは重要なのです。',
    'たぶん大丈夫でしょう。',
    '今日はちょっと疲れたね。',
]
for source in tests:
    print(source, '->', to_nida(source))


## 4. `原文 -> ニダ口調` の JSONL を生成

In [ ]:
written = 0
with OUTPUT_PATH.open('w', encoding='utf-8') as output:
    for source in corpus:
        source = source.strip()
        if not source or len(source) < 2 or len(source) > 256:
            continue
        target = to_nida(source)
        if not target or target == source:
            continue
        output.write(json.dumps({'style': 'NIDA_FICTION', 'source': source, 'target': target}, ensure_ascii=False) + '\n')
        written += 1
        if written >= MAX_ROWS:
            break
print('written:', written)
print('output:', OUTPUT_PATH)
